So, since we've tokenized our data let's load it immediately and perform a simple safe check to be sure everything is as it should be.

In [2]:
from datasets import load_from_disk
dataset = load_from_disk("../data/processed/tokenized_codexglue")

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 10000
    })
    valid: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 2000
    })
})


In [5]:
print(dataset['train'][0])

{'input_ids': [14, 20, 3, 16, 6, 15, 11, 2, 4, 10, 15, 11, 15, 5, 68069, 17, 11, 16, 7, 19, 3, 15, 4, 18, 17, 12, 8, 13, 5, 15, 5, 17, 12, 9, 13, 7, 19, 3, 68068, 4, 12, 8, 13], 'labels': [3, 5, 12, 7, 9, 13, 14, 8, 4, 2, 10014, 6, 11, 10, 10015]}


In [6]:
print(dataset['valid'][0])

{'input_ids': [14, 0, 3, 255, 6, 42, 11, 48, 4, 10, 36, 42, 98, 48, 10, 42, 11, 41, 7, 42, 7, 100, 3, 1374, 7, 47759, 3, 4, 6, 0, 4, 132, 3666, 7, 30959, 3, 4, 28, 7601, 10, 0, 3, 41, 7, 42, 7, 100, 3, 7601, 6, 31706, 4, 4, 0, 11, 41, 7, 42, 7, 100, 3, 7601, 6, 0, 4, 132, 20151, 7, 18806, 3, 0, 6, 665, 4, 28, 20159, 10, 89, 1421, 6, 6175, 6, 1586, 96, 41, 7, 6185, 3, 7601, 4, 10, 89, 1433, 96, 1586, 10, 7391, 11, 41, 7, 42, 7, 100, 3, 1421, 6, 1433, 4, 36, 7391, 488, 0, 10, 20159, 7, 1199, 3, 7391, 6, 41, 7, 42, 7, 16423, 3, 7391, 6, 7601, 4, 4, 132, 58, 3, 0, 6, 6514, 4, 28, 523, 10, 0, 11, 523, 7, 172, 3, 4, 132, 58, 3, 42, 6, 6519, 4, 28, 523, 10, 0, 7, 2843, 3, 3, 0, 6, 255, 7, 0, 4, 6, 523, 4], 'labels': [2148, 825, 41, 4, 1852, 535, 178, 34]}


In [7]:
print(dataset['test'][0])

{'input_ids': [14, 0, 3, 11241, 4, 10, 0, 11, 12, 13, 3941, 11, 29351, 3, 11241, 4, 89, 679, 96, 3941, 7, 26746, 3, 0, 4, 10, 2207, 11, 679, 7, 26746, 3, 1163, 4, 12, 8, 13, 0, 7, 200, 3, 2207, 7, 3940, 12, 8, 13, 7, 236, 4, 18, 0], 'labels': [670, 10014, 767, 349, 570, 1420, 41, 943, 1344, 10015, 3816, 0, 10015]}


For starters we're gonna build a simple seq2seq model, without attention (we'll then add it later). We're gonna start with a RNN, since, as explained in [this paper](https://www.arxiv.org/pdf/1909.04352v1#:~:text=Recurrent%20neural%20network%20is%20most,a%20special%20kind%20of%20RNN), RNN seems the more convenient choice between that and CNN. 

They also say that LSTM used with RNN produced great results, so that's the way we're gonna take for now.

In [8]:
import torch.nn as nn

In [ ]:

class Encoder(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.hidden = None # final hidden state
        self.n_outputs = n_outputs
        self.cell = None
        self.basic_rnn = nn.LSTM(self.n_features, self.hidden_dim, batch_first=True) # NLF

    def forward(self, X):
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(X) # NLH, 1NH, 1NH
        return batch_first_output, (self.hidden, self.cell)